# **RNN Forward Propagation**

Just as in ANN we do z = (wx + b), likewise in RNN we do z = ((w_1 * x) + (w_2 * h_t-1) + b).<br>
- **IMP :** The weights in RNN remain same at each step for, but the wight for current input and the weight for the previous output will be different btwn themselves.
- Such as the weights for current input will be high to priotize it, and the weight of previous outputs will be relatively less to give it less importance during training.
- `layers.SimpelRNN()` - contains multiple time steps (usually 10 time steps), with in these time steps the weights will remain same, but weights btwn different `layers.SimpleRNN()` layers in a RNN neural network may change.

**REMEMBER : If we send 32 batch of row data into the layer 1, each neuron in layer 1 will get the 32 z values. And all the 32 z values will be sent into the activation function, and hence we get the 32 `a` values.**<br><br>
**Now after getting the 32 `a` values we will send them into the each neuron in next layer and do z = (wa + b) for all 32 `a` values and then again they will be sent to the activation function. This process goes on until the output layer.**

---

### **Core Intuition**

Suppose we have an RNN hidden layer with **6 neurons (hidden state of size 6)** followed by an output layer with **1 neuron** to predict the final value $\hat{y}$ (a Many-to-One architecture).


#### 1. Forward Pass Through Time (Steps 1 to 10)

* **Step 1 ($t = 1$):**  
  At the start of training, the model's weights and biases are randomly initialized. Each of the 6 neurons receives the external input for step 1 ($x_1$) plus an initial hidden state ($h_0$ (weights), usually initialized to zeros). Each neuron calculates its weighted sum, adds its bias, applies an activation function (like $\tanh$), and produces its hidden state output ($h_1$).

* **Step 2 ($t = 2$):**  
  To produce the output at step 2, each neuron needs two things:
  1. The new external input at step 2 ($x_2$).
  2. The hidden state outputs from step 1 ($t-1$) of **all 6 neurons** (including its own previous output).
  
  Each neuron multiplies these inputs and previous hidden states by their respective weights, adds a bias, and passes the sum through an activation function to generate its new output for step 2 ($h_2$).
  
  > **Note:** The weights used here are the **exact same shared weights** used in step 1.

* **Steps 3 to 10:**  
  This sequential recurrent process continues all the way through **step 10**.


#### 2. Generating the Prediction ($\hat{y}$)

At step 10, the final hidden state outputs from all 6 neurons ($h_{10}$) are sent forward to the final output layer (1 neuron). 
* These 6 outputs are multiplied by **6 new weights ($W_{hy}$)** and a bias is added.
* The output neuron computes the final prediction **$\hat{y}$**.
* The loss/error is calculated by comparing $\hat{y}$ with the actual target value $y$:
  $$\text{Loss} = \mathcal{L}(\hat{y}, y)$$


#### 3. Backpropagation Through Time (BPTT)

Once the loss is computed, backpropagation begins to minimize the error:
1. Gradients flow backwards from the output layer into the hidden state at **step 10**.
2. Because step 10's output depended on **step 9**, gradients flow back into step 9, then step 8, all the way back to **step 1**.
3. Gradients from all 10 steps are accumulated to update the **shared weights and biases** using an optimizer (like Adam or SGD).

This entire cycle (Forward pass $\to$ Loss calculation $\to$ Backpropagation Through Time $\to$ Weight update) repeats for many epochs until the error is minimized close to 0.

---

During a single backpropagation step, two distinct sets of weights are updated:

**RNN_Layer_Calculation (At each step t):**<br>
$$output_t = tanh((U * x_t) + (W11 * output1Prev) + (W12 * output2Prev) + ... + (W16 * output6Prev) + b)$$

**Output_Layer_Calculation (At the final step):**<br>
$$y_hat = sigmoid((V1 * output1Step10) + (V2 * output2Step10) + ... + (V6 * output6Step10) + b)$$

### **Eg.**<br>
Neuron 1 to neuron 6 are being assigned their weights as 1,2,3,4,5, and 6, now it is like **"output = (1.input ) + (2.output2) + (3.output3) .... (6.ouput6) + b"**.<br>
Now these weights remain same throught the 10 steps, until they are being changed in backpropagation.

The input data in RNN is (time_stamps, input features)

Lets take this dataset as example

![img](https://miro.medium.com/v2/resize:fit:638/format:webp/1*ga8-P7rdErT4qswAJiwQYQ.png)

Our vocabulary contains 5 words which are : [movie, was, good, bad, not]

So now movie can be represented by [1,0,0,0,0], was can be represented by [0,1,0,0,0], good can be represented by [0,0,1,0,0], bad can be represented by [0,0,0,1,0], not can be represented by [0,0,0,0,1]

So the sentence ‘movie was good’ can be written in vector form as :

[[1,0,0,0,0],[0,1,0,0,0],[0,0,1,0,0]]

So now one by one we send the word to our RNN, which means at timestamp t = 1 the word ‘movie’ will be sent, at timestamp t = 2 the word ‘was’ will be sent, at timestamp t = 3 the word ‘good’ will be sent.

So, for sentence 1 (“movie was good”) the shape of the input data is (3,5) where 3 is number of timestamps, and 5 denotes the total number of features

For sentence 2 the shape of the input data will be (3,5) and for sentence 3 the shape of the input data will be (4,5)

While writing the code, when we will use keras -> SimpleRNN (class); we will send the data in this format **(batch_size, timestamps, input features)**. Lets say if we send all the 3 sentences at a time, so the data which will be sent to RNN will be (3, 4, 5) where 4 denotes timestamps for the longest sentence in the dataset. So this 3D tensor will be sent to our RNN.

## **RNN Architecture**

From now on, we will call sentence 1 as X1, sentence 2 as X2, and sentence 3 as X3. 1st word of X1 will be called as X11, 2nd word of X1 will called as X12 and 3rd word of X1 will be called as X13. The same logic will apply for X2, X3. 

RNN is very similar to ANN. Only 2 major differences are there

- In RNN, we send the input based on time
    - At t = 1, we send X11 to RNN
    - At t = 2, we send X12 to RNN… so on
- ANN is a feed forward neural network, which means information moves only in the forward direction that is from input to output

But an RNN structure looks like this

![img](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*XaRviNRyUFxbBle6QlOA5g.png)

Lets draw the RNN architecture.

There need to be 5 neurons in the input layer as each word is going to be represented as vector of 5 numbers. Now for the hidden layer lets put 3 neurons in it.

But since these 3 neurons are recurring units, their output will be input to themselves. The output of each of the 3 neurons or recurring units will be sent to other neurons in the same layer as well

![img](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*a7bF1U1UVOsTWd3XbOueuA.png)

![img](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*xekk0lYOu_VkBALzN1hZ1g.jpeg)

The total number of weights will be $5*3 + 3*3 + 3 = 27$ (5*3 between input and recurring units, 3*3 between recurring units themselves, 3 between recurring units and the output). The total number of bias will be 4 (3 in the hidden layer that is for each recurring unit and one for output node).

## **Forward propagation in RNN**

![img](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*ZhbUqsP-cFb30H5LMLFd7A.png)

Note : X11, X12 are all vectors and each of these vectors are 5 dimensional (since there are 5 unique words in our corpus, each word will be represented as a vector of these 5 words).

We will be sending these words one by one to our RNN

![img](https://miro.medium.com/v2/resize:fit:1394/format:webp/1*PfCBkl71FCgyiA1rq9x0hQ.png)

While doing forward propagation we follow a concept called unfolding through time which is nothing but the recurrent layer works as a loop. **Every node in the recurrent layer has an activation function which by default is tanh.**

![img](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*DJ-is6pleS7v2l0US8NAAQ.png)<br>
At t=1, the box over here represents the recurrent layer, Wi is the representation of all the weights from the input to the recurrent layer


<br><br>
![img](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*19EoCsPVsS_JITen3u0nWA.png)<br>
At t=2


<br><br>
![img](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*x1vgB4bhNQHH2-lPg9M5Rg.png)<br>
Here O2 is the output of the activation function when X12.Wi + O1.Wh + b is being sent as input to it


<br><br>
![img](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*SVe49yX0EfDuAH3txLmq3Q.png)<br>
At t = 3<br>
To maintain consistency, at t = 1 we sent O0 which is a vector of (1x3) which will either have all 0 values or random numbers in it

<br><br>
![img](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*V6LuOeNLFu_fGd4VVd8ryg.png)<br>
Overall picture

## **Why are recurrent neural networks named as recurrent?**

Because the hidden layer is recurring or we can say that the input is changing word by word, but we are using this layer everytime to calculate the output.

Also, everytime we are giving a new input to our RNN, but we are using the same weight values so basically the concept of parameter sharing or weight sharing is being used
A simple RNN can process a sequence of 10 time steps